# Chapter 26 Lab: End-to-End Forecasting Pipeline

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-26-end-to-end-forecasting-pipeline-lab.ipynb)

This lab builds a complete forecasting workflow from raw ordered data to a forecast report. The purpose is not to find the most complicated model. The purpose is to make every modeling decision explicit, testable, and reproducible.

By the end of this lab, you should be able to:

1. write a forecasting contract with target, frequency, forecast origin, horizon, information available at the origin, and loss function;
2. audit a raw time series for missing values, irregular time stamps, outliers, and possible external variables;
3. construct lag, rolling, calendar, and known-future features without using future target values;
4. compare baselines and machine-learning models using chronological validation;
5. run rolling-origin validation;
6. create simple residual-based forecast intervals;
7. package a final forecast table and model card;
8. use AI assistance as a skeptical reviewer, not as an unverified authority.

Throughout the lab, keep the forecasting rule in mind:

$$
\widehat{Y}_{t+h|t} = g_h(F_t),
$$

where $F_t$ means all information available at forecast origin $t$. A feature is valid only if it belongs to $F_t$.

## 1. Imports and reproducibility

We use only standard Colab-friendly packages. The models are intentionally small so that the full lab can run quickly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

rng = np.random.default_rng(7339)


def rmse(y_true, y_pred):
    """Root mean squared error."""
    return mean_squared_error(y_true, y_pred) ** 0.5


def mase(y_true, y_pred, insample, seasonality=7):
    """Mean absolute scaled error using a seasonal naive denominator."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    insample = np.asarray(insample, dtype=float)
    denom = np.mean(np.abs(insample[seasonality:] - insample[:-seasonality]))
    if denom == 0:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / denom

print("Packages imported.")

## 2. Simulate a realistic daily-demand data set

We create a daily demand series with trend, weekly seasonality, annual seasonality, promotions, holiday effects, price effects, temperature effects, noise, missing observations, and outliers.

The column `y_observed` represents what the analyst receives. The column `y_true_for_lab` is kept only so that we can check the simulation. In a real project, the true clean signal is not available.

In [ ]:
n = 3 * 365
dates = pd.date_range("2023-01-01", periods=n, freq="D")
t = np.arange(n)
day = dates.dayofweek.to_numpy()
year_phase = 2 * np.pi * t / 365.25

weekly = 8 * np.sin(2 * np.pi * day / 7 - 0.8)
yearly = 12 * np.sin(year_phase - 0.5)
trend = 0.035 * t

holiday = (
    ((dates.month == 11) & (dates.day >= 23) & (dates.day <= 29))
    | ((dates.month == 12) & (dates.day <= 3))
)

promo = np.zeros(n, dtype=int)
promo_dates = pd.to_datetime([
    "2023-03-15", "2023-06-10", "2023-11-25",
    "2024-03-20", "2024-07-05", "2024-11-29",
    "2025-02-14", "2025-08-28",
])
for d in promo_dates:
    promo[(dates >= d) & (dates <= d + pd.Timedelta(days=2))] = 1

price = 10.0 + 0.0015 * t - 0.8 * promo + 0.2 * np.sin(year_phase)
temp_forecast = 55 + 18 * np.sin(year_phase - 1.1) + rng.normal(0, 3.0, n)
noise = rng.normal(0, 4.0, n)

y_true = (
    90
    + trend
    + weekly
    + yearly
    + 15 * promo
    + 10 * holiday.astype(float)
    - 2.2 * price
    + 0.08 * temp_forecast
    + noise
)

y_observed = y_true.copy()
missing_idx = rng.choice(np.arange(50, n - 50), size=20, replace=False)
y_observed[missing_idx] = np.nan
outlier_idx = rng.choice(np.arange(80, n - 80), size=8, replace=False)
y_observed[outlier_idx] += rng.choice([95, -70], size=len(outlier_idx))

df = pd.DataFrame({
    "ds": dates,
    "y_observed": y_observed,
    "y_true_for_lab": y_true,
    "promo": promo,
    "holiday": holiday.astype(int),
    "price": price,
    "temp_forecast": temp_forecast,
})

print(df.head())
print("Number of rows:", len(df))

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(df["ds"], df["y_observed"], linewidth=1)
plt.title("Raw observed daily demand")
plt.xlabel("date")
plt.ylabel("observed demand")
plt.show()

## 3. Audit the raw data

Before modeling, ask basic questions.

- Are dates unique and sorted?
- Are any observations missing?
- Are the time gaps regular?
- Are there values that are impossible under domain knowledge?
- Which variables are known at forecast time, and which variables must themselves be forecast?

For this lab, we assume `promo`, `holiday`, `price`, and `temp_forecast` are known or forecasted before the demand forecast is produced. In a real project, that assumption must be documented.

In [ ]:
audit = pd.DataFrame({
    "check": [
        "date_min", "date_max", "rows", "unique_dates",
        "missing_y", "minimum_gap", "maximum_gap", "y_min", "y_max",
    ],
    "value": [
        df["ds"].min(),
        df["ds"].max(),
        len(df),
        df["ds"].nunique(),
        int(df["y_observed"].isna().sum()),
        df["ds"].diff().dropna().min(),
        df["ds"].diff().dropna().max(),
        float(np.nanmin(df["y_observed"])),
        float(np.nanmax(df["y_observed"])),
    ]
})

audit

In [ ]:
# A simple domain rule: demand below 20 or above 200 is flagged as suspicious.
flagged = df[(df["y_observed"] < 20) | (df["y_observed"] > 200) | (df["y_observed"].isna())]
flagged[["ds", "y_observed", "promo", "holiday", "price"]].head(12)

## 4. Clean the target with explicit rules

The rules below are deliberately simple and documented:

1. values below 20 or above 200 are treated as impossible demand records;
2. impossible values are set to missing;
3. missing values are repaired using past information through forward fill;
4. if the first record were missing, we would use the first later nonmissing value only to initialize the series.

In a real production pipeline, cleaning thresholds and imputation rules should be justified by domain knowledge and tested on historical data.

In [ ]:
def repair_target_past_first(series, lower=20, upper=200):
    repaired = series.copy()
    impossible = (repaired < lower) | (repaired > upper)
    repaired[impossible] = np.nan
    repaired = repaired.ffill().bfill()
    return repaired


df["y_clean"] = repair_target_past_first(df["y_observed"])
print("Missing values after repair:", int(df["y_clean"].isna().sum()))
print(df[["ds", "y_observed", "y_clean"]].head())

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(df["ds"], df["y_observed"], linewidth=1, label="observed")
plt.plot(df["ds"], df["y_clean"], linewidth=1.2, label="cleaned")
plt.title("Observed and cleaned demand")
plt.xlabel("date")
plt.ylabel("demand")
plt.legend()
plt.show()

## 5. Write the forecasting contract

A forecasting contract makes the statistical task precise. The contract below says exactly what is being forecast and what information is allowed at forecast time.

In [ ]:
forecasting_contract = {
    "target": "daily cleaned demand y_clean",
    "frequency": "daily",
    "forecast_origin": "end of each day after demand is observed",
    "horizon": "1 to 14 days ahead",
    "information_available": [
        "past cleaned demand values up to the origin",
        "calendar variables for future dates",
        "planned promotions",
        "planned or forecasted price",
        "forecasted temperature",
    ],
    "point_metrics": ["MAE", "RMSE", "MASE"],
    "interval_metric": "empirical coverage of a residual-based interval",
    "main_validation_design": "chronological train/validation/test plus rolling-origin checks",
}

forecasting_contract

## 6. Build leakage-safe features

A valid feature row for date $t$ may use $Y_{t-1},Y_{t-2},\ldots$ but not $Y_t$ or future values of the target. Calendar variables and planned variables for date $t$ may be used only if they are known at the forecast origin.

The function below constructs one-step-ahead supervised rows. The target is `y_clean` at date $t$. All lag and rolling features are shifted so they use only dates before $t$.

In [ ]:
def add_calendar_features(data):
    out = data.copy()
    date_parts = out["ds"].dt
    dow = date_parts.dayofweek.to_numpy()
    doy = date_parts.dayofyear.to_numpy()

    out["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    out["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    out["annual_sin"] = np.sin(2 * np.pi * doy / 365.25)
    out["annual_cos"] = np.cos(2 * np.pi * doy / 365.25)
    out["is_weekend"] = (dow >= 5).astype(int)
    out["time_index"] = np.arange(len(out))
    return out


def make_features(data, y_col="y_clean", lags=(1, 2, 7, 14, 28), rolling_windows=(7, 14, 28)):
    base = add_calendar_features(data)

    out = base[[
        "ds", "promo", "holiday", "price", "temp_forecast",
        "dow_sin", "dow_cos", "annual_sin", "annual_cos",
        "is_weekend", "time_index",
    ]].copy()

    y = base[y_col]

    for lag in lags:
        out[f"lag_{lag}"] = y.shift(lag)

    shifted_y = y.shift(1)
    for window in rolling_windows:
        out[f"roll_mean_{window}"] = shifted_y.rolling(window).mean()
        out[f"roll_std_{window}"] = shifted_y.rolling(window).std()
        out[f"roll_min_{window}"] = shifted_y.rolling(window).min()
        out[f"roll_max_{window}"] = shifted_y.rolling(window).max()

    out["target"] = y
    return out.dropna().reset_index(drop=True)


feature_df = make_features(df)
feature_cols = [c for c in feature_df.columns if c not in ["ds", "target"]]

print("Feature table shape:", feature_df.shape)
print("Number of features:", len(feature_cols))
feature_df.head()

## 7. Unit-test the timing of a feature

A pipeline should contain small tests that catch leakage. The following test verifies that `roll_mean_7` for a selected target date is the mean of the previous seven cleaned target values, not a mean that includes the target date.

In [ ]:
row = 20
check_date = feature_df.loc[row, "ds"]
original_position = int(df.index[df["ds"] == check_date][0])
manual = df.loc[original_position - 7:original_position - 1, "y_clean"].mean()
computed = feature_df.loc[row, "roll_mean_7"]

print("date checked:", check_date.date())
print("manual previous-seven mean:", round(manual, 6))
print("feature value:", round(computed, 6))
assert np.isclose(manual, computed)

## 8. Chronological train/validation/test split

We use the earliest part for training, the next part for validation and model selection, and the most recent part for final testing. The test set is not used until the model choice has been made.

In [ ]:
last_date = feature_df["ds"].max()
test_start = last_date - pd.Timedelta(days=119)
val_start = test_start - pd.Timedelta(days=119)

train_df = feature_df[feature_df["ds"] < val_start].copy()
val_df = feature_df[(feature_df["ds"] >= val_start) & (feature_df["ds"] < test_start)].copy()
test_df = feature_df[feature_df["ds"] >= test_start].copy()

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "start": [train_df["ds"].min(), val_df["ds"].min(), test_df["ds"].min()],
    "end": [train_df["ds"].max(), val_df["ds"].max(), test_df["ds"].max()],
    "rows": [len(train_df), len(val_df), len(test_df)],
})

split_summary

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(train_df["ds"], train_df["target"], label="train", linewidth=1)
plt.plot(val_df["ds"], val_df["target"], label="validation", linewidth=1)
plt.plot(test_df["ds"], test_df["target"], label="test", linewidth=1)
plt.title("Chronological split")
plt.xlabel("date")
plt.ylabel("cleaned demand")
plt.legend()
plt.show()

## 9. Baselines

Every forecasting pipeline needs simple baselines. A machine-learning model that cannot beat a seasonal-naive forecast is probably not useful.

In [ ]:
def summarize_forecast(name, y_true, y_pred, insample):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MASE": mase(y_true, y_pred, insample, seasonality=7),
    }


y_val = val_df["target"].to_numpy()
insample_train = train_df["target"].to_numpy()

baseline_val = pd.DataFrame([
    summarize_forecast("naive_lag_1", y_val, val_df["lag_1"], insample_train),
    summarize_forecast("seasonal_naive_lag_7", y_val, val_df["lag_7"], insample_train),
    summarize_forecast("rolling_mean_7", y_val, val_df["roll_mean_7"], insample_train),
    summarize_forecast("rolling_mean_28", y_val, val_df["roll_mean_28"], insample_train),
])

baseline_val.sort_values("MAE")

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(val_df["ds"], val_df["target"], label="actual", linewidth=1.4)
plt.plot(val_df["ds"], val_df["lag_7"], label="seasonal naive", linewidth=1)
plt.plot(val_df["ds"], val_df["roll_mean_28"], label="rolling mean 28", linewidth=1)
plt.title("Validation baselines")
plt.xlabel("date")
plt.ylabel("demand")
plt.legend()
plt.show()

## 10. Candidate models inside reproducible pipelines

Scaling and imputation are fitted on the training split only. The `Pipeline` object helps enforce that rule.

In [ ]:
model_registry = {
    "ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ]),
    "random_forest": RandomForestRegressor(
        n_estimators=30,
        min_samples_leaf=5,
        random_state=7339,
        n_jobs=1,
    ),
    "gradient_boosting": GradientBoostingRegressor(
        n_estimators=50,
        learning_rate=0.05,
        max_depth=2,
        random_state=7339,
    ),
}


def evaluate_model(model, train_part, eval_part):
    fitted = clone(model)
    fitted.fit(train_part[feature_cols], train_part["target"])
    pred = fitted.predict(eval_part[feature_cols])
    metrics = summarize_forecast(
        name="temporary",
        y_true=eval_part["target"],
        y_pred=pred,
        insample=train_part["target"],
    )
    return fitted, pred, metrics

model_val_records = []
val_predictions = pd.DataFrame({"ds": val_df["ds"], "actual": val_df["target"]})
fitted_on_train = {}

for name, model in model_registry.items():
    fitted, pred, metrics = evaluate_model(model, train_df, val_df)
    fitted_on_train[name] = fitted
    val_predictions[name] = pred
    metrics["model"] = name
    model_val_records.append(metrics)

model_val = pd.DataFrame(model_val_records)
validation_results = pd.concat([baseline_val, model_val], ignore_index=True).sort_values("MAE")
validation_results

In [ ]:
best_model_name = validation_results.iloc[0]["model"]
print("Best validation model:", best_model_name)

# If a baseline wins, we still keep the best machine-learning model for the later sections.
ml_results = model_val.sort_values("MAE")
best_ml_name = ml_results.iloc[0]["model"]
print("Best machine-learning model:", best_ml_name)

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(val_predictions["ds"], val_predictions["actual"], label="actual", linewidth=1.4)
for name in model_registry:
    plt.plot(val_predictions["ds"], val_predictions[name], label=name, linewidth=1)
plt.title("Validation forecasts from candidate models")
plt.xlabel("date")
plt.ylabel("demand")
plt.legend()
plt.show()

## 11. Final holdout test after model selection

Now that we have chosen the best machine-learning model using the validation set, we retrain it on train plus validation and evaluate it once on the final test set.

In [ ]:
trainval_df = pd.concat([train_df, val_df], ignore_index=True)
selected_model = clone(model_registry[best_ml_name])
selected_model.fit(trainval_df[feature_cols], trainval_df["target"])

test_pred = selected_model.predict(test_df[feature_cols])
test_metrics = pd.DataFrame([
    summarize_forecast("selected_" + best_ml_name, test_df["target"], test_pred, trainval_df["target"]),
    summarize_forecast("seasonal_naive_lag_7", test_df["target"], test_df["lag_7"], trainval_df["target"]),
    summarize_forecast("rolling_mean_28", test_df["target"], test_df["roll_mean_28"], trainval_df["target"]),
]).sort_values("MAE")

test_metrics

In [ ]:
test_forecast_df = pd.DataFrame({
    "ds": test_df["ds"],
    "actual": test_df["target"],
    "forecast": test_pred,
    "seasonal_naive": test_df["lag_7"],
})

test_forecast_df["residual"] = test_forecast_df["actual"] - test_forecast_df["forecast"]

test_forecast_df.head()

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(test_forecast_df["ds"], test_forecast_df["actual"], label="actual", linewidth=1.4)
plt.plot(test_forecast_df["ds"], test_forecast_df["forecast"], label="selected model", linewidth=1)
plt.plot(test_forecast_df["ds"], test_forecast_df["seasonal_naive"], label="seasonal naive", linewidth=1)
plt.title("Final holdout test forecasts")
plt.xlabel("date")
plt.ylabel("demand")
plt.legend()
plt.show()

## 12. Residual diagnostics

A residual plot is not a proof that the model is correct, but it often reveals problems: bias, remaining seasonality, changing variance, outliers, or regime changes.

In [ ]:
def sample_acf(values, max_lag=28):
    values = np.asarray(values, dtype=float)
    centered = values - values.mean()
    denom = np.sum(centered ** 2)
    acfs = []
    for lag in range(max_lag + 1):
        if lag == 0:
            acfs.append(1.0)
        else:
            acfs.append(np.sum(centered[lag:] * centered[:-lag]) / denom)
    return np.asarray(acfs)

resid = test_forecast_df["residual"].to_numpy()
print("Residual mean:", round(resid.mean(), 4))
print("Residual standard deviation:", round(resid.std(ddof=1), 4))
print("Residual MAE:", round(np.mean(np.abs(resid)), 4))

plt.figure(figsize=(9, 3.2))
plt.axhline(0, linewidth=1)
plt.plot(test_forecast_df["ds"], resid, marker="o", markersize=3, linewidth=1)
plt.title("Test residuals over time")
plt.xlabel("date")
plt.ylabel("actual minus forecast")
plt.show()

acf_vals = sample_acf(resid, max_lag=28)
plt.figure(figsize=(8, 3.2))
plt.stem(np.arange(len(acf_vals)), acf_vals)
plt.axhline(0, linewidth=1)
plt.title("Sample ACF of test residuals")
plt.xlabel("lag")
plt.ylabel("ACF")
plt.show()

## 13. Rolling-origin validation

A single split can be lucky or unlucky. Rolling-origin validation repeatedly trains on earlier data and tests on a later forecast origin.

The first rolling function checks one-step-ahead forecasts. Later, we will extend this idea to a 14-day recursive forecast.

In [ ]:
def rolling_one_step_scores(feature_data, feature_cols, model, min_train=400, step=14):
    records = []
    for end in range(min_train, len(feature_data) - 1, step):
        train_part = feature_data.iloc[:end]
        test_row = feature_data.iloc[[end]]

        fitted = clone(model)
        fitted.fit(train_part[feature_cols], train_part["target"])
        pred = float(fitted.predict(test_row[feature_cols])[0])
        actual = float(test_row["target"].iloc[0])

        records.append({
            "ds": test_row["ds"].iloc[0],
            "actual": actual,
            "forecast": pred,
            "error": actual - pred,
        })
    return pd.DataFrame(records)

rolling_records = {}
rolling_summary_records = []

for name, model in model_registry.items():
    roll = rolling_one_step_scores(feature_df, feature_cols, model, min_train=560, step=35)
    rolling_records[name] = roll
    rolling_summary_records.append({
        "model": name,
        "rolling_MAE": mean_absolute_error(roll["actual"], roll["forecast"]),
        "rolling_RMSE": rmse(roll["actual"], roll["forecast"]),
    })

rolling_summary = pd.DataFrame(rolling_summary_records).sort_values("rolling_MAE")
rolling_summary

In [ ]:
plt.figure(figsize=(9, 3.5))
for name, roll in rolling_records.items():
    plt.plot(roll["ds"], roll["error"], marker="o", markersize=3, linewidth=1, label=name)
plt.axhline(0, linewidth=1)
plt.title("Rolling one-step forecast errors")
plt.xlabel("date")
plt.ylabel("error")
plt.legend()
plt.show()

## 14. Recursive 14-day forecasting

One-step models are often used recursively: predict tomorrow, append that prediction to the history, predict the next day, and so on. This can accumulate error, so horizon-wise evaluation is important.

In [ ]:
def one_feature_row(date, row_exog, history_values, time_index):
    """Build one feature row for a future date using only the available history."""
    date = pd.Timestamp(date)
    dow = date.dayofweek
    doy = date.dayofyear

    vals = {
        "promo": float(row_exog["promo"]),
        "holiday": float(row_exog["holiday"]),
        "price": float(row_exog["price"]),
        "temp_forecast": float(row_exog["temp_forecast"]),
        "dow_sin": np.sin(2 * np.pi * dow / 7),
        "dow_cos": np.cos(2 * np.pi * dow / 7),
        "annual_sin": np.sin(2 * np.pi * doy / 365.25),
        "annual_cos": np.cos(2 * np.pi * doy / 365.25),
        "is_weekend": int(dow >= 5),
        "time_index": int(time_index),
    }

    for lag in (1, 2, 7, 14, 28):
        vals[f"lag_{lag}"] = history_values[-lag] if len(history_values) >= lag else np.nan

    for window in (7, 14, 28):
        recent = np.asarray(history_values[-window:], dtype=float)
        vals[f"roll_mean_{window}"] = np.nanmean(recent) if len(recent) else np.nan
        vals[f"roll_std_{window}"] = np.nanstd(recent, ddof=1) if len(recent) > 1 else 0.0
        vals[f"roll_min_{window}"] = np.nanmin(recent) if len(recent) else np.nan
        vals[f"roll_max_{window}"] = np.nanmax(recent) if len(recent) else np.nan

    return pd.DataFrame([vals])[feature_cols]


def recursive_forecast(model, past_data, future_data):
    history = past_data["y_clean"].astype(float).tolist()
    forecasts = []
    for _, row in future_data.iterrows():
        x_row = one_feature_row(row["ds"], row, history, time_index=len(history))
        pred = float(model.predict(x_row)[0])
        forecasts.append(pred)
        history.append(pred)
    return np.asarray(forecasts)

print("Recursive forecast helper is ready.")

In [ ]:
# Example: forecast the first 14 days of the final test period from the day before the test begins.
test_origin = test_df["ds"].min() - pd.Timedelta(days=1)
past_for_origin = df[df["ds"] <= test_origin].copy()
future_14 = df[(df["ds"] > test_origin) & (df["ds"] <= test_origin + pd.Timedelta(days=14))].copy()

forecast_14 = recursive_forecast(selected_model, past_for_origin, future_14)
example_14 = pd.DataFrame({
    "origin": test_origin,
    "ds": future_14["ds"].to_numpy(),
    "horizon": np.arange(1, len(future_14) + 1),
    "actual": future_14["y_clean"].to_numpy(),
    "forecast": forecast_14,
})
example_14["error"] = example_14["actual"] - example_14["forecast"]
example_14.head(14)

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(example_14["horizon"], example_14["actual"], marker="o", label="actual")
plt.plot(example_14["horizon"], example_14["forecast"], marker="o", label="recursive forecast")
plt.title("A 14-day recursive forecast from one origin")
plt.xlabel("horizon")
plt.ylabel("demand")
plt.legend()
plt.show()

## 15. Rolling-origin multi-step backtest

Now we repeat the 14-day recursive forecast for several origins. This reveals whether error grows with the horizon.

In [ ]:
def rolling_multistep_backtest(raw_data, feature_data, feature_cols, model, origins, horizon=14, min_train_rows=420):
    records = []
    for origin in origins:
        train_part = feature_data[feature_data["ds"] <= origin].copy()
        if len(train_part) < min_train_rows:
            continue

        future = raw_data[(raw_data["ds"] > origin) & (raw_data["ds"] <= origin + pd.Timedelta(days=horizon))].copy()
        if len(future) < horizon:
            continue

        fitted = clone(model)
        fitted.fit(train_part[feature_cols], train_part["target"])

        past = raw_data[raw_data["ds"] <= origin].copy()
        preds = recursive_forecast(fitted, past, future)

        for h, (date, actual, pred) in enumerate(zip(future["ds"], future["y_clean"], preds), start=1):
            records.append({
                "origin": origin,
                "ds": date,
                "horizon": h,
                "actual": float(actual),
                "forecast": float(pred),
                "error": float(actual - pred),
            })
    return pd.DataFrame(records)

origins = pd.date_range(val_start, last_date - pd.Timedelta(days=15), freq="56D")
multistep_bt = rolling_multistep_backtest(
    raw_data=df,
    feature_data=feature_df,
    feature_cols=feature_cols,
    model=model_registry[best_ml_name],
    origins=origins,
    horizon=14,
)

print("Backtest rows:", len(multistep_bt))
multistep_bt.head()

In [ ]:
horizon_records = []
for h, group in multistep_bt.groupby("horizon"):
    horizon_records.append({
        "horizon": h,
        "MAE": mean_absolute_error(group["actual"], group["forecast"]),
        "RMSE": rmse(group["actual"], group["forecast"]),
        "mean_error": group["error"].mean(),
    })

horizon_summary = pd.DataFrame(horizon_records)
horizon_summary

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(horizon_summary["horizon"], horizon_summary["MAE"], marker="o", label="MAE")
plt.plot(horizon_summary["horizon"], horizon_summary["RMSE"], marker="o", label="RMSE")
plt.title("Error by forecast horizon")
plt.xlabel("horizon")
plt.ylabel("error")
plt.legend()
plt.show()

## 16. Residual-based forecast intervals

A simple interval can be formed from validation residual quantiles. This is not a full probabilistic model, but it is transparent and easy to audit.

The interval below uses the selected model fitted on the training split, computes residual quantiles on the validation split, then applies those offsets to the final test forecasts.

In [ ]:
interval_model = clone(model_registry[best_ml_name])
interval_model.fit(train_df[feature_cols], train_df["target"])
val_pred_for_interval = interval_model.predict(val_df[feature_cols])
val_resid = val_df["target"].to_numpy() - val_pred_for_interval

lo_offset, hi_offset = np.quantile(val_resid, [0.05, 0.95])

interval_df = test_forecast_df.copy()
interval_df["lower_90"] = interval_df["forecast"] + lo_offset
interval_df["upper_90"] = interval_df["forecast"] + hi_offset
interval_df["covered"] = (
    (interval_df["actual"] >= interval_df["lower_90"])
    & (interval_df["actual"] <= interval_df["upper_90"])
)

coverage = interval_df["covered"].mean()
print("Nominal interval level: 0.90")
print("Empirical test coverage:", round(coverage, 3))
interval_df.head()

In [ ]:
plt.figure(figsize=(9, 3.6))
plt.plot(interval_df["ds"], interval_df["actual"], label="actual", linewidth=1.3)
plt.plot(interval_df["ds"], interval_df["forecast"], label="forecast", linewidth=1.2)
plt.fill_between(
    interval_df["ds"],
    interval_df["lower_90"].to_numpy(),
    interval_df["upper_90"].to_numpy(),
    alpha=0.2,
    label="90% residual interval",
)
plt.title("Test forecasts with residual interval")
plt.xlabel("date")
plt.ylabel("demand")
plt.legend()
plt.show()

## 17. Produce a deployment-style 14-day forecast

Finally, fit the selected model on all available feature rows and produce a 14-day forecast beyond the last observed date. In real deployment, the future table would come from a calendar, promotion plan, price plan, and weather forecast.

In [ ]:
final_model = clone(model_registry[best_ml_name])
final_model.fit(feature_df[feature_cols], feature_df["target"])

future_dates = pd.date_range(df["ds"].max() + pd.Timedelta(days=1), periods=14, freq="D")
future_t = np.arange(len(df), len(df) + len(future_dates))
future_phase = 2 * np.pi * future_t / 365.25
future_promo = np.zeros(len(future_dates), dtype=int)
future_promo[[5, 6, 7]] = 1
future_holiday = (((future_dates.month == 12) & (future_dates.day <= 3))).astype(int)
future_price = 10.0 + 0.0015 * future_t - 0.8 * future_promo + 0.2 * np.sin(future_phase)
future_temp = 55 + 18 * np.sin(future_phase - 1.1)

future_plan = pd.DataFrame({
    "ds": future_dates,
    "y_observed": np.nan,
    "y_true_for_lab": np.nan,
    "promo": future_promo,
    "holiday": future_holiday,
    "price": future_price,
    "temp_forecast": future_temp,
    "y_clean": np.nan,
})

final_forecast = recursive_forecast(final_model, df, future_plan)

# Use validation residual offsets for a transparent interval around the deployment forecast.
deployment_report = pd.DataFrame({
    "forecast_origin": df["ds"].max(),
    "ds": future_dates,
    "horizon": np.arange(1, 15),
    "model": best_ml_name,
    "target": "daily cleaned demand",
    "forecast": final_forecast,
    "lower_90": final_forecast + lo_offset,
    "upper_90": final_forecast + hi_offset,
    "promo": future_promo,
    "holiday": future_holiday,
    "price": future_price,
    "temp_forecast": future_temp,
})

deployment_report

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(deployment_report["horizon"], deployment_report["forecast"], marker="o", label="forecast")
plt.fill_between(
    deployment_report["horizon"],
    deployment_report["lower_90"].to_numpy(),
    deployment_report["upper_90"].to_numpy(),
    alpha=0.2,
    label="90% interval",
)
plt.title("Deployment-style 14-day forecast")
plt.xlabel("horizon")
plt.ylabel("demand")
plt.legend()
plt.show()

## 18. Save the forecast report and write a model card

A forecast table should be accompanied by enough metadata for another analyst to understand how it was produced.

In [ ]:
report_path = "lab26_forecast_report.csv"
deployment_report.to_csv(report_path, index=False)
print("Saved:", report_path)

model_card = f"""
Model card: Chapter 26 forecasting pipeline

Target: {forecasting_contract['target']}
Frequency: {forecasting_contract['frequency']}
Horizon: {forecasting_contract['horizon']}
Selected machine-learning model: {best_ml_name}
Validation design: chronological validation plus rolling-origin checks
Training period for final model: {feature_df['ds'].min().date()} to {feature_df['ds'].max().date()}
Final test period: {test_df['ds'].min().date()} to {test_df['ds'].max().date()}
Features: lags, rolling summaries, calendar variables, promotions, holidays, price, temperature forecast
Point metrics on final test set: see test_metrics table
Interval method: validation residual quantile offsets
Known limitations: synthetic data; simple residual interval; assumed future exogenous variables are available; recursive forecasts can accumulate error
Monitoring plan: track residual mean, rolling MAE, interval coverage, missingness, feature drift, and changes in promotion or pricing policy
""".strip()

print(model_card)

## 19. AI-assisted pipeline review prompt

Use AI as a skeptical reviewer. Do not ask it to approve your work. Ask it to find failure modes and verification steps.

Copy, edit, and use the following prompt after your own pipeline is running:

> I am building a time-series forecasting pipeline. The target is daily demand. The frequency is daily. The forecast horizon is 1 to 14 days ahead. The information available at the forecast origin includes past demand, calendar variables, planned promotions, planned price, and forecasted temperature. My preprocessing repairs impossible target values and forward-fills missing values. My features include lags, rolling means, rolling standard deviations, calendar Fourier features, promotions, holidays, price, and temperature. My validation design uses chronological train/validation/test splitting and rolling-origin evaluation. My baselines are naive, seasonal naive, and rolling means. My candidate models are ridge regression, random forest, and gradient boosting. My metrics are MAE, RMSE, MASE, horizon-wise errors, and interval coverage. Please act as a skeptical reviewer. Identify possible leakage, horizon mismatch, weak baselines, invalid transformations, nonstationarity, calibration problems, exogenous-variable assumptions, and unsupported conclusions. For every concern, give a concrete verification step I can implement in code.

## 20. Exercises

1. Change the forecast horizon from 14 days to 28 days. Which part of the pipeline changes?
2. Add a direct multi-step model that trains a separate model for each horizon. Compare it with the recursive strategy.
3. Remove the promotion feature. How much does validation and test performance change?
4. Replace the residual interval with split conformal intervals using the validation residuals.
5. Add a drift monitor: compute rolling MAE over the test period using a 28-day window.
6. Create a leakage bug intentionally by using `rolling(window).mean()` without `shift(1)`. Show how the timing unit test catches the problem.
7. Write a model card for a real data set from your field.

## Mini-project

Choose a real daily, weekly, or monthly time series. Build the same pipeline:

1. forecasting contract;
2. data audit;
3. leakage-safe features;
4. baselines;
5. candidate models;
6. chronological validation;
7. rolling-origin validation;
8. forecast intervals;
9. final forecast report;
10. model card and AI-assisted critique.